<a href="https://colab.research.google.com/github/Mvic11/AP005-3-2026-3/blob/main/Corte1/Tarea3/udmy17_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Agrupación de Datos en Pandas (groupby)En este cuaderno se explora el objeto DataFrameGroupBy mediante el método .groupby(), el cual implementa el flujo de trabajo Split-Apply-Combine (Dividir-Aplicar-Combinar) para analizar métricas agregadas por sectores en el conjunto de datos fortune1000.csv.
1. Conexión a Drive, Carga e Inspección
Montamos Google Drive, leemos el archivo asignando la columna Rank como índice primario e inspeccionamos la estructura general del DataFrame.

In [ ]:
import pandas as pd
from google.colab import drive

# Montaje de Google Drive
drive.mount('/content/drive')

# Carga del dataset utilizando 'Rank' como índice
fortune = pd.read_csv("/content/drive/MyDrive/udmy/dataAnalisys/fortune1000.csv", index_col="Rank")

# Inspección inicial
fortune.head(3)
fortune.info()

2. Creación e Inspección del Objeto GroupBy
Al ejecutar fortune.groupby('Sector'), Pandas no genera una nueva tabla visible inmediatamente, sino un objeto de tipo DataFrameGroupBy que agrupa las filas según las categorías únicas de la columna especificada:

sector.size(): Devuelve una Serie con el número de registros en cada grupo (similar a value_counts()).

sector.first() / sector.last(): Devuelve la primera y última fila de cada sector respetando el orden del conjunto de datos.

sector.get_group("Retailing"): Extrae un DataFrame con las empresas pertenecientes exclusivamente a ese grupo. Es equivalente a aplicar un filtro booleano (fortune[fortune["Sector"] == "Retailing"]).

In [ ]:
# Creación del objeto groupby
sector = fortune.groupby('Sector')

# Verificación del tipo de objeto
type(sector)

# Conteo de elementos por sector
sector.size()

# Inspección del primer y último registro por sector
sector.first()
sector.last()

# Extracción de un grupo específico
sector.get_group("Retailing")

3. Agregaciones Estadísticas (sum, mean, agg)
Podemos aplicar funciones estadísticas directamente sobre todo el objeto agrupado, sobre columnas numéricas específicas o mediante el método flexible .agg():

sector["Employees"].sum(): Suma el total de empleados por sector.

sector["Employees"].mean(): Calcula el promedio de empleados por sector.

sector.agg(...): Permite definir un diccionario para aplicar distintas funciones de agregación a diferentes columnas simultáneamente (e.g., calcular el promedio y el máximo de ingresos, así como el máximo y mínimo de ganancias).

In [ ]:
# Suma y promedio sobre una columna específica
sector["Employees"].sum()
sector["Employees"].mean()

# Agregaciones avanzadas personalizadas por columna
sector.agg({
    "Revenues": ["mean", "max"],
    "Profits": ["mean", "max", "min"],
    "Employees": "sum"
})

4. Iteración sobre Grupos
Es posible iterar a través de las parejas (nombre_grupo, dataframe_grupo) que genera el objeto agrupado. En este ejemplo, extraemos la empresa con mayor ganancia de cada sector y consolidamos los resultados en un nuevo DataFrame mediante pd.concat().

In [ ]:
# Lista para almacenar los DataFrames filtrados por sector
highest_profit_list = []

# Iteración sobre cada sector y sus filas asociadas
for sector_name, group_df in sector:
    company = group_df.nlargest(1, "Profits")
    highest_profit_list.append(company)

# Consolidador final mediante pd.concat
top_companies_df = pd.concat(highest_profit_list)
top_companies_df